In [1]:
import os 


In [2]:
%pwd

'c:\\Users\\LENOVO\\Kidney-Disease-Classification-DL-Project-DVC\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\LENOVO\\Kidney-Disease-Classification-DL-Project-DVC'

In [5]:
from dataclasses import dataclass 
from pathlib import Path 

@dataclass(frozen=True)
class PrepareBaseModelConfig:
	root_dir: Path 
	base_model_path: Path 
	updated_base_model_path: Path 
	params_image_size: list 
	params_learning_rate: float
	params_include_top: bool
	params_weights: str 
	params_classes: int 

In [6]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories

In [9]:
class ConfigurationManager:
	def __init__(self, config_filepath=CONFIG_FILE_PATH, params_filepath=PARAMS_FILE_PATH):
		self.config = read_yaml(config_filepath)
		self.params = read_yaml(params_filepath)
		create_directories([self.config.artifacts_root])

In [10]:
from cnnClassifier.constants import * 
from cnnClassifier.utils.common import read_yaml, create_directories 

In [ ]:
class ConfigurationManager:
	def __init__(self, config_filepath=CONFIG_FILE_PATH, params_filepath=PARAMS_FILE_PATH):
		self.config = read_yaml(config_filepath)
		self.params = read_yaml(params_filepath)
		create_directories([self.config.artifacts_root])
	
	def get_prepare_base_model_config(self)-> PrepareBaseModelConfig:
		config = self.config.prepare_base_model
		params = self.params.get("TrainingArguments", self.params.get("training_arguments", self.params))
		prepare_base_model_config = PrepareBaseModelConfig(
			root_dir=Path(config.root_dir),
			base_model_path=Path(config.base_model_path),
			updated_base_model_path=Path(config.updated_base_model_path),
			params_image_size=params.get("image_size", params.get("IMAGE_SIZE")),
			params_learning_rate=params.get("learning_rate", params.get("LEARNING_RATE")),
			params_include_top=params.get("include_top", params.get("INCLUDE_TOP")),
			params_weights=params.get("weights", params.get("WEIGHTS")),
			params_classes=params.get("classes", params.get("CLASSES"))
		)
		return prepare_base_model_config
	

In [31]:
import os 
import urllib.request as request 
from zipfile import ZipFile 
import tensorflow as tf 

In [33]:
class PrepareBaseModel:
	def __init__(self, config: PrepareBaseModelConfig):
		self.config = config
		self.model = None
		self.full_model = None

	def get_base_model(self):
		# use the correct constructor for VGG16 and ensure image size is a tuple
		self.model = tf.keras.applications.VGG16(
			input_shape=tuple(self.config.params_image_size),
			weights=self.config.params_weights,
			include_top=self.config.params_include_top,
		)
		self.save_model(path=self.config.base_model_path, model=self.model)
		return self.model

	@staticmethod
	def _prepare_full_model(model, classes, freeze_all, freeze_till, learning_rate):
		if freeze_all:
			for layer in model.layers:
				layer.trainable = False
		elif (freeze_till is not None) and (freeze_till > 0):
			for layer in model.layers[:-freeze_till]:
				layer.trainable = False
		
		flatten_layer = tf.keras.layers.Flatten()(model.output)
		prediction = tf.keras.layers.Dense(units=classes, activation="softmax")(flatten_layer)
		

		full_model = tf.keras.models.Model(inputs=model.input, outputs=prediction)
		full_model.compile(
			optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
			loss=tf.keras.losses.CategoricalCrossentropy(),
			metrics=["accuracy"],
		)
		full_model.summary()
		return full_model

	def update_base_model(self):
		if self.model is None:
			raise RuntimeError("Base model not initialized. Call get_base_model() first.")
		self.full_model = self._prepare_full_model(
			model=self.model,
			classes=self.config.params_classes,
			freeze_all=True,
			freeze_till=None,
			learning_rate=self.config.params_learning_rate,
		)
		self.save_model(path=self.config.updated_base_model_path, model=self.full_model)
		return self.full_model

	@staticmethod
	def save_model(path: Path, model):
		model.save(str(path))

In [34]:
try:
	config = ConfigurationManager()
	prepare_base_model_config = config.get_prepare_base_model_config()
	prepare_base_model = PrepareBaseModel(config=prepare_base_model_config)
	prepare_base_model.get_base_model()
	prepare_base_model.update_base_model()
except Exception as e:
	print(f"Error: {e}")

[2026-06-10 02:10:01,409: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-06-10 02:10:01,415: INFO: common: yaml file: params.yaml loaded successfully]
[2026-06-10 02:10:01,420: INFO: common: created directory at: artifacts]
58889256/58889256 [==============================] - 47s 1us/step
[2026-06-10 02:10:50,037: WARNING: saving_utils: Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.]
Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 224, 224, 3)]     0         
                                                                 
 block1_conv1 (Conv2D)       (None, 224, 224, 64)      1792      
                                                                 
 block1_conv2 (Conv2D)       (None, 224, 224, 64)      36928     
        